# Notebook 04 - TFT: Pre-Training & Fine-Tuning

**Wichtig:** Fuer Phase 2 (Fine-Tuning) werden nur Planungsraeume mit
VOLLSTAENDIGER Zeitreihe verwendet - identisch zu Notebook 03. plr_id
wird konsequent auf 8-stellige Strings normalisiert (zfill).

In [1]:
# ═══════════════════════════════════════════════════════════════
# ZELLE 1: Imports & Meta-Informationen laden
# ═══════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import pickle
import shutil
import warnings
warnings.filterwarnings('ignore')

from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.metrics import MAE, RMSE
from lightning.pytorch import Trainer
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint

torch.manual_seed(42)
print('✅ Imports erfolgreich!')

with open('../models/meta.pkl', 'rb') as f:
    meta = pickle.load(f)
SEQ_LEN = meta.get('seq_len', 3)
ALLE_JAHRE_META = [int(y) for y in meta.get('jahre', [])]
print(f'✅ Meta geladen: SEQ_LEN={SEQ_LEN}, Anker-Jahre={ALLE_JAHRE_META}')

✅ Imports erfolgreich!
✅ Meta geladen: SEQ_LEN=3, Anker-Jahre=[2013, 2015, 2017, 2019, 2021, 2023]


In [2]:
# ═══════════════════════════════════════════════════════════════
# PHASE 1: PRE-TRAINING Dataset (synthetisch)
# ═══════════════════════════════════════════════════════════════

df_synth = pd.read_csv('../data/raw/mss_pretraining_synthetisch.csv')
df_synth['plr_id'] = df_synth['plr_id'].astype(str).str.zfill(8)

FEATURE_COLS_SYNTH = [
    'arbeitslosigkeit', 'sgb2_quote', 'kinderarmut',
    'dist_zentrum', 'status_lag'
]
TARGET = 'status_next'

years_synth = sorted(df_synth['year'].unique())
cutoff_synth = df_synth[df_synth['year'] <= years_synth[-4]]['time_idx'].max()

pretrain_ds = TimeSeriesDataSet(
    df_synth[df_synth['time_idx'] <= cutoff_synth],
    time_idx='time_idx',
    target=TARGET,
    group_ids=['plr_id'],
    max_encoder_length=SEQ_LEN,
    max_prediction_length=1,
    time_varying_known_reals=['time_idx'],
    time_varying_unknown_reals=FEATURE_COLS_SYNTH,  # OPTION B: TARGET entfernt
    static_reals=['dist_zentrum'],
    target_normalizer=None,
    add_relative_time_idx=True,
    allow_missing_timesteps=True,
)

pretrain_val_ds = TimeSeriesDataSet.from_dataset(
    pretrain_ds, df_synth, predict=True, stop_randomization=True)

pretrain_train_dl = pretrain_ds.to_dataloader(train=True, batch_size=64, num_workers=0)
pretrain_val_dl   = pretrain_val_ds.to_dataloader(train=False, batch_size=64, num_workers=0)

print(f'✅ Pre-Training Dataset: {len(pretrain_ds)} Samples')

tft_pretrain = TemporalFusionTransformer.from_dataset(
    pretrain_ds,
    learning_rate=0.001,
    hidden_size=32,
    lstm_layers=2,
    attention_head_size=2,
    dropout=0.2,
    hidden_continuous_size=16,
    loss=MAE(),
)
print(f'✅ TFT Modell: {sum(p.numel() for p in tft_pretrain.parameters()):,} Parameter')

✅ Pre-Training Dataset: 9214 Samples
✅ TFT Modell: 89,777 Parameter


In [3]:
# ═══════════════════════════════════════════════════════════════
# Pre-Training durchfuehren
# ═══════════════════════════════════════════════════════════════

checkpoint_pt = ModelCheckpoint(
    dirpath='../models/', filename='tft_pretrained',
    monitor='val_loss', mode='min', save_top_k=1,
)
early_stop_pt = EarlyStopping(monitor='val_loss', patience=10, mode='min')

trainer_pt = Trainer(
    max_epochs=80, accelerator='mps', devices=1,
    gradient_clip_val=0.1,
    callbacks=[early_stop_pt, checkpoint_pt],
    enable_progress_bar=True, log_every_n_steps=5,
)

print('🚀 TFT Pre-Training startet...')
trainer_pt.fit(tft_pretrain, pretrain_train_dl, pretrain_val_dl)
print(f'✅ Pre-Training abgeschlossen! Bestes Modell: {trainer_pt.checkpoint_callback.best_model_path}')

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


🚀 TFT Pre-Training startet...



   | Name                               | Type                            | Params | Mode  | FLOPs
--------------------------------------------------------------------------------------------------------
0  | loss                               | MAE                             | 0      | train | 0    
1  | logging_metrics                    | ModuleList                      | 0      | train | 0    
2  | input_embeddings                   | MultiEmbedding                  | 0      | train | 0    
3  | prescalers                         | ModuleDict                      | 224    | train | 0    
4  | static_variable_selection          | VariableSelectionNetwork        | 1.8 K  | train | 0    
5  | encoder_variable_selection         | VariableSelectionNetwork        | 14.0 K | train | 0    
6  | decoder_variable_selection         | VariableSelectionNetwork        | 3.8 K  | train | 0    
7  | static_context_variable_selection  | GatedResidualNetwork            | 4.3 K  | train | 0    
8  

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

... (42 outputs left out)

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

✅ Pre-Training abgeschlossen! Bestes Modell: /Users/lyu/Desktop/geoXAI_berlin/models/tft_pretrained-v2.ckpt


In [4]:
# ═══════════════════════════════════════════════════════════════
# PHASE 2a: Echte Daten laden & auf VOLLSTAENDIGE Zeitreihen filtern
# ═══════════════════════════════════════════════════════════════
# Nutzt df_full.csv (nicht train+val+test zusammengesetzt!) fuer
# vollstaendige Encoder-Historie inkl. reiner Historie-Jahre.

train   = pd.read_csv('../data/final/train.csv')
train['plr_id'] = train['plr_id'].astype(str).str.zfill(8)
val     = pd.read_csv('../data/final/val.csv')
val['plr_id'] = val['plr_id'].astype(str).str.zfill(8)
test    = pd.read_csv('../data/final/test.csv')
test['plr_id'] = test['plr_id'].astype(str).str.zfill(8)
df_full_raw = pd.read_csv('../data/final/df_full.csv')
df_full_raw['plr_id'] = df_full_raw['plr_id'].astype(str).str.zfill(8)

with open('../models/feature_cols.pkl', 'rb') as f:
    FEATURE_COLS = pickle.load(f)
with open('../models/target.pkl', 'rb') as f:
    TARGET = pickle.load(f)

FEATURE_COLS_COMMON = [f for f in FEATURE_COLS if f in FEATURE_COLS_SYNTH]
print(f'Gemeinsame Features: {FEATURE_COLS_COMMON}')

df_all_raw = pd.concat([train, val, test], ignore_index=True)
df_all_raw = df_all_raw.sort_values(['plr_id', 'year']).reset_index(drop=True)

required_years = set(ALLE_JAHRE_META) if ALLE_JAHRE_META else set(df_all_raw['year'].unique())
jahre_pro_plr = df_full_raw.groupby('plr_id')['year'].apply(lambda s: set(int(y) for y in s))
vollstaendige_plr = jahre_pro_plr[jahre_pro_plr.apply(lambda s: required_years.issubset(s))].index

df_all = df_full_raw[df_full_raw['plr_id'].isin(vollstaendige_plr)].copy()

print(f'\n📊 Vollstaendigkeits-Check:')
print(f'   Benoetigte Anker-Jahre: {sorted(required_years)}')
print(f'   PLR gesamt:             {df_full_raw["plr_id"].nunique()}')
print(f'   PLR mit vollstaendiger Zeitreihe: {len(vollstaendige_plr)}')

year_map = {y: i for i, y in enumerate(sorted(df_all['year'].unique()))}
df_all['time_idx'] = df_all['year'].map(year_map)

train_years_set = set(int(y) for y in train['year'].unique())
training_cutoff = df_all[df_all['year'].isin(train_years_set)]['time_idx'].max()

print(f'\nTrain-Jahre (aus train.csv): {sorted(train_years_set)}')
print(f'training_cutoff (time_idx): {training_cutoff}')

if pd.isna(training_cutoff) or len(df_all) == 0:
    raise RuntimeError(
        'Keine vollstaendigen PLR fuer Training verfuegbar! '
        'Notebook 02 (Areal-Interpolation) pruefen.'
    )

Gemeinsame Features: ['arbeitslosigkeit', 'sgb2_quote', 'kinderarmut', 'dist_zentrum', 'status_lag']

📊 Vollstaendigkeits-Check:
   Benoetigte Anker-Jahre: [2013, 2015, 2017, 2019, 2021, 2023]
   PLR gesamt:             541
   PLR mit vollstaendiger Zeitreihe: 519

Train-Jahre (aus train.csv): [2019]
training_cutoff (time_idx): 3


In [5]:
# ═══════════════════════════════════════════════════════════════
# PHASE 2b: TFT Datasets bauen (auf gefiltertem df_all)
# ═══════════════════════════════════════════════════════════════

ft_train_ds = TimeSeriesDataSet(
    df_all[df_all['time_idx'] <= training_cutoff],
    time_idx='time_idx', target=TARGET, group_ids=['plr_id'],
    max_encoder_length=SEQ_LEN, max_prediction_length=1,
    time_varying_known_reals=['time_idx'],
    time_varying_unknown_reals=FEATURE_COLS_COMMON,  # OPTION B: TARGET entfernt
    static_reals=['dist_zentrum'],
    target_normalizer=None, add_relative_time_idx=True,
    allow_missing_timesteps=True,
)

val_time_idx = df_all[df_all['year'].isin(val['year'].unique())]['time_idx'].max()
ft_val_ds = TimeSeriesDataSet.from_dataset(
    ft_train_ds, df_all[df_all['time_idx'] <= val_time_idx],
    predict=True, stop_randomization=True)

ft_test_ds = TimeSeriesDataSet.from_dataset(
    ft_train_ds, df_all, predict=True, stop_randomization=True)

ft_train_dl = ft_train_ds.to_dataloader(train=True, batch_size=32, num_workers=0)
ft_val_dl   = ft_val_ds.to_dataloader(train=False, batch_size=32, num_workers=0)
ft_test_dl  = ft_test_ds.to_dataloader(train=False, batch_size=32, num_workers=0)

print(f'✅ Fine-Tuning Dataset: Train={len(ft_train_ds)} Val={len(ft_val_ds)} Test={len(ft_test_ds)}')

if len(ft_train_ds) == 0:
    raise RuntimeError(
        f'Fine-Tuning Train-Set ist leer! SEQ_LEN={SEQ_LEN}, '
        f'benoetigte Jahre={sorted(required_years)}. '
        'Bitte pruefen ob genug vollstaendige PLR vorhanden sind.'
    )

✅ Fine-Tuning Dataset: Train=519 Val=519 Test=519


In [6]:
# ═══════════════════════════════════════════════════════════════
# Fine-Tuning: Pre-Training-Gewichte laden + nachjustieren
# ═══════════════════════════════════════════════════════════════

best_pretrain_path = trainer_pt.checkpoint_callback.best_model_path

if FEATURE_COLS_COMMON == FEATURE_COLS_SYNTH:
    tft_finetuned = TemporalFusionTransformer.load_from_checkpoint(best_pretrain_path)
    print('✅ Pre-Training-Gewichte als Startpunkt geladen!')
else:
    print('⚠️ Feature-Set unterscheidet sich vom Pre-Training - TFT wird neu initialisiert')
    tft_finetuned = TemporalFusionTransformer.from_dataset(
        ft_train_ds, learning_rate=0.0001, hidden_size=32,
        lstm_layers=2, attention_head_size=2, dropout=0.2,
        hidden_continuous_size=16, loss=MAE(),
    )

tft_finetuned.hparams.learning_rate = 0.0001

checkpoint_ft = ModelCheckpoint(
    dirpath='../models/', filename='tft_finetuned',
    monitor='val_loss', mode='min', save_top_k=1,
)
early_stop_ft = EarlyStopping(monitor='val_loss', patience=8, mode='min')

trainer_ft = Trainer(
    max_epochs=40, accelerator='mps', devices=1,
    gradient_clip_val=0.1,
    callbacks=[early_stop_ft, checkpoint_ft],
    enable_progress_bar=True, log_every_n_steps=5,
)

print('🚀 TFT Fine-Tuning startet...')
trainer_ft.fit(tft_finetuned, ft_train_dl, ft_val_dl)

best_ft_path = trainer_ft.checkpoint_callback.best_model_path
print(f'✅ Fine-Tuning abgeschlossen!')
print(f'   Finales Modell: {best_ft_path}')

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.

   | Name                               | Type                            | Params | Mode  | FLOPs
--------------------------------------------------------------------------------------------------------
0  | loss                               | MAE                             | 0      | train | 0    
1  | logging_metrics                    | ModuleList                      | 0      | train | 0    
2  | input_embeddings                   | MultiEmbedding                  | 0      | train | 0    
3  | prescalers                         | ModuleDict                      | 224    | train | 0    
4  | static_variable_selection          | VariableSelectionNetwork        | 1.8 K  | train

✅ Pre-Training-Gewichte als Startpunkt geladen!
🚀 TFT Fine-Tuning startet...


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

... (36 outputs left out)

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=40` reached.


✅ Fine-Tuning abgeschlossen!
   Finales Modell: /Users/lyu/Desktop/geoXAI_berlin/models/tft_finetuned-v2.ckpt


In [7]:
# ═══════════════════════════════════════════════════════════════
# Finalen Modellpfad sichern (fuer spaetere Notebooks)
# ═══════════════════════════════════════════════════════════════

shutil.copy(best_ft_path, '../models/tft_finetuned_final.ckpt')

with open('../models/tft_checkpoint_path.pkl', 'wb') as f:
    pickle.dump('../models/tft_finetuned_final.ckpt', f)

with open('../models/tft_feature_cols_common.pkl', 'wb') as f:
    pickle.dump(FEATURE_COLS_COMMON, f)

print('✅ NOTEBOOK 04 ABGESCHLOSSEN!')
print('\n📁 Gespeicherte Modelle:')
print(f'   Pre-Training: {best_pretrain_path}')
print(f'   Fine-Tuned:   models/tft_finetuned_final.ckpt  <- FINALES Modell')
print('\n→ Weiter mit Notebook 05 (Evaluierung)')

✅ NOTEBOOK 04 ABGESCHLOSSEN!

📁 Gespeicherte Modelle:
   Pre-Training: /Users/lyu/Desktop/geoXAI_berlin/models/tft_pretrained-v2.ckpt
   Fine-Tuned:   models/tft_finetuned_final.ckpt  <- FINALES Modell

→ Weiter mit Notebook 05 (Evaluierung)
